In [2]:
!pip install pandas_gbq==0.33.0 -qq
!pip install numpy==1.26.4 -qq
!pip install cloudscraper==1.2.71 -qq
!pip install beautifulsoup4==4.12.2 -qq
!pip install google-cloud-bigquery==3.40.0 -qq

In [3]:
import os
import csv
import time
import random
import shutil
import tempfile
import re
import glob
from datetime import date, timedelta, datetime
import io
import pandas_gbq
import numpy as np
import pandas as pd
import cloudscraper
from bs4 import BeautifulSoup
from google.cloud import bigquery
from urllib.parse import urlparse, parse_qs, unquote

In [4]:
# Path lakehouse (thay bằng lakehouse bạn đã attach), ví dụ '/lakehouse/default'
LAKEHOUSE_PATH = os.environ.get("LAKEHOUSE_PATH", "/lakehouse/default")
# Thư mục con dùng cho project
ADS_DATA_DIR = os.path.join(LAKEHOUSE_PATH, "Files", "ads_data")

BATCH_SAVE_SIZE = 100
REQUEST_TIMEOUT = 15

# --- Logging setup (viết cả vào Lakehouse) ---
os.makedirs(ADS_DATA_DIR, exist_ok=True)

In [5]:
# --- Utilities: cloudscraper session with retry ---

def create_session():
    print("Creating cloudscraper session")
    s = cloudscraper.create_scraper()
    return s

In [6]:
import time
import random

def get_max_page_with_retries(base,
                              attempts=5,
                              initial_delay=1.0,
                              backoff_factor=2.0,
                              jitter=0.3,
                              default_on_failure=1,
                              verbose=True):
    """
    Thử fetch trang và parse max page tối đa `attempts` lần.
    Sử dụng exponential backoff + jitter. Trả về default_on_failure khi tất cả thất bại.
    """
    delay = initial_delay
    last_exc = None

    for attempt in range(1, attempts + 1):
        try:
            soup, _ = soup_from_url(base.format(i=2))
            max_page = get_max_page_from_list_soup(soup)
            # nếu parser trả về None/không hợp lệ, coi là lỗi
            if not max_page or not isinstance(max_page, int):
                raise ValueError(f"Invalid max_page value: {max_page!r}")
            if verbose:
                print(f"For base {base}, determined max_page={max_page} (attempt {attempt})")
            return max_page

        except Exception as e:
            last_exc = e
            if attempt == attempts:
                msg = (f"All {attempts} attempts failed. Returning default {default_on_failure}. "
                       f"Last error: {e!r}")
                print(msg)
                return default_on_failure

            # in lỗi và sleep với jitter
            sleep_for = delay + random.uniform(0, jitter)
            if verbose:
                print(f"Attempt {attempt}/{attempts} failed: {e!s} — retrying in {sleep_for:.2f}s")
            time.sleep(sleep_for)
            delay *= backoff_factor

    # phòng trường hợp không tới được
    return default_on_failure

In [7]:
def get_with_retry(url, max_attempts=5, timeout=REQUEST_TIMEOUT):
    for attempt in range(1, max_attempts + 1):
        try:
            resp = session.get(url, timeout=timeout)
            status = resp.status_code
            # print("GET %s attempt=%d status=%s elapsed=%.2fs", url, attempt, status, elapsed)

            if resp.status_code == 200:
                return resp
            else:
                if resp.status_code in (429, 500, 502, 503, 504):
                    # print("Retryable status %s for %s (attempt %d)", status, url, attempt)
                    time.sleep(1 * attempt)
                    continue
                else:
                    return resp
        except Exception as e:
            print(f"GET {url} attempt={attempt} failed: {e}")
            if attempt == max_attempts:
                # print(f"Max attempts reached for {url}, raising exception")
                raise
            time.sleep(0.5 * attempt)
    raise Exception("Failed to GET url: " + url)

In [8]:
# --- HTML parsing helpers ---

def text_or_none(el):
    if not el:
        return None
    return el.get_text(strip=True)


def parse_date_from_element(elem):
    if not elem:
        return None
    val = elem.get("aria-label") or elem.get_text(strip=True)
    val = val.strip()
    for fmt in ("%d/%m/%Y", "%d/%m/%y", "%Y-%m-%d"):
        try:
            return datetime.strptime(val, fmt).date()
        except Exception:
            pass
    m = re.search(r"(\d{1,2}/\d{1,2}/\d{2,4})", val)
    if m:
        try:
            return datetime.strptime(m.group(1), "%d/%m/%Y").date()
        except:
            try:
                return datetime.strptime(m.group(1), "%d/%m/%y").date()
            except:
                pass
    return None


def extract_ads_id_from_link(link):
    try:
        m = re.search(r"(\d+)(?:\D*$)", link)
        if m:
            return int(m.group(1))
    except:
        pass
    return None

In [9]:
def get_existed_ads_ids(project_id="real-estate-project-445516", dataset_id="real_estate_data", table_id="ads_data"):
    client = bigquery.Client()

    # Construct the SQL query
    query = f"""
        SELECT DISTINCT [Mã tin]
        FROM `{project_id}.{dataset_id}.{table_id}`
    """

    try:
        # Run the query
        query_job = client.query(query)
        result = query_job.result()

        # Extract all values
        ads_ids = [row.Ma_tin for row in result]
        return ads_ids
    except Exception as e:
        print(f"Error get_existed_ads_ids: {e}")
        return None


In [10]:
def find_start_page(base_url_template, end_date, low, high):
    # binary search on pages by checking dates
    while low < high:
        mid = (low + high) // 2
        url = base_url_template.format(i=mid)
        try:
            soup, _ = soup_from_url(url)
        except Exception:
            return low
        cards = (soup.find(id="product-lists-web") or soup).select(".js__card-full-web")
        if not cards:
            cards = (soup.find(id="product-lists-web") or soup).select(".re__card--item")
        # if no cards, break
        if not cards:
            return low
        # check published date of first card
        date_elem = cards[0].select_one("span.re__card-published-info-published-at")
        ngay_gia_han = parse_date_from_element(date_elem)
        if ngay_gia_han is None:
            # fallback check last card
            date_elem2 = cards[-1].select_one("span.re__card-published-info-published-at")
            ngay_gia_han = parse_date_from_element(date_elem2)
        if ngay_gia_han is None:
            return low
        if ngay_gia_han <= end_date:
            high = mid
        else:
            low = mid + 1
    return low

In [11]:
def scrape_links_from_base_phase1(base_url_template, start_date, end_date, max_page, existed_ads_ids_sorted=None):
    all_links = []
    flag = existed_ads_ids_sorted is not None
    find_cards_retry = 5

    for i in range(1, max_page):
        # print("Page ", i)
        url = base_url_template.format(i=i)
        try:
            soup, resp = soup_from_url(url)
        except Exception as e:
            print(f"Failed to fetch page {url}: {e}")
            continue

        product_list = soup.find(id="product-lists-web")
        if not product_list:
            # sometimes listing structure different; try common card selectors
            product_list = soup.select_one(".product-listing") or soup
            if not product_list:
                print("product_list not found") 
                raise
                
        cards = product_list.select(".js__card-full-web")[:50]  # take up to 50
        if not cards:
            # alternative selector
            cards = product_list.select(".re__card--item")[:50]
            if not cards:
                if find_cards_retry > 0:
                    find_cards_retry = find_cards_retry - 1
                    continue
                print(f"cards not found at url: {url}")
                raise

        for card in cards:
            date_elem = card.select_one("span.re__card-published-info-published-at")
            if not date_elem:
                print("date_elem not found")
                raise
                
            label = card.find("div", class_="label--v1")
            if label is None:
                print(f"Phase 1 found {len(all_links)} links.")
                return all_links, i

            ngay_gia_han = parse_date_from_element(date_elem)
            if not ngay_gia_han:
                # fallback: try time tag or small text
                date_elem2 = card.find("time") or card.select_one(".card__time")
                ngay_gia_han = parse_date_from_element(date_elem2)

            if not ngay_gia_han:
                print("ngay_gia_han not found")
                raise

            if ngay_gia_han > end_date or ngay_gia_han < start_date:
                # print("ngay_gia_han not yesterday, continue")
                continue

            # get link
            a = card.find("a", href=True)
            if not a:
                print("not a, raise")
                raise

            link = a["href"]
            if not link.startswith("http"):
                link = "https://batdongsan.com.vn" + link

            ads_id = extract_ads_id_from_link(link)
            if flag and ads_id is not None:
                # check with bisect logic; simple membership test (list is small)
                # use binary search:
                import bisect
                idx = bisect.bisect_left(existed_ads_ids_sorted, ads_id)
                if idx < len(existed_ads_ids_sorted) and existed_ads_ids_sorted[idx] == ads_id:
                    # already exists -> skip
                    continue
            all_links.append((link, ngay_gia_han))

    # deduplicate
    all_links = list(set(all_links))
    print(f"scrape_links_from_base -> found {len(all_links)} links")
    return all_links

In [12]:
# --- Scrape links (list pages) ---

def get_max_page_from_list_soup(soup):
    try:
        pag = soup.find("div", class_="re__pagination-group")
        if pag is None:
            raise
        texts = [x.get_text(strip=True) for x in pag.find_all("a")[:-1]]
        nums = [int(re.sub(r"\.", "", t)) for t in texts if re.sub(r"\.", "", t).isdigit()]
        if nums:
            return max(nums)
    except Exception as e:
        print(f"Error finding max page: {e}")
        raise
    return 1


def soup_from_url(url):
    resp = get_with_retry(url)
    
    try:
        soup = BeautifulSoup(resp.text, "html.parser")
        return soup, resp
    except Exception as e:
        print("Failed to parse HTML for {url}: {e}")
        raise


def scrape_links_from_base_phase2(base_url_template, start_page, start_date, end_date, max_page, existed_ads_ids_sorted=None):
    all_links = []
    flag = existed_ads_ids_sorted is not None
    find_cards_retry = 5
    retry = 10

    for i in range(max(1, start_page - 2), max_page + 1):
        url = base_url_template.format(i=i)
        try:
            soup, resp = soup_from_url(url)
        except Exception as e:
            print("Failed to fetch page {url}: {e}")
            continue

        product_list = soup.find(id="product-lists-web")
        if not product_list:
            product_list = soup.select_one(".product-listing") or soup

        cards = product_list.select(".js__card-full-web")[:50]
        if not cards:
            cards = product_list.select(".re__card--item")[:50]
        if not cards:
            if find_cards_retry > 0:
                find_cards_retry = find_cards_retry - 1
                continue
            print(f"cards not found at url: {url}")
            raise

        for card in cards:
            date_elem = card.select_one("span.re__card-published-info-published-at")
            ngay_gia_han = parse_date_from_element(date_elem)
            if not ngay_gia_han:
                date_elem2 = card.find("time") or card.select_one(".card__time")
                ngay_gia_han = parse_date_from_element(date_elem2)

            if not ngay_gia_han:
                continue

            if ngay_gia_han > end_date:
                continue
            if ngay_gia_han < start_date:
                if retry > 0:
                    retry = retry - 1
                    break
                print(f"scrape_links_from_base -> reached older date {ngay_gia_han.isoformat()} < {start_date.isoformat()} at page {i}, stopping.")
                return all_links

            a = card.find("a", href=True)
            if not a:
                continue
            link = a["href"]
            if not link.startswith("http"):
                link = "https://batdongsan.com.vn" + link

            ads_id = extract_ads_id_from_link(link)
            if flag and ads_id is not None:
                import bisect
                idx = bisect.bisect_left(existed_ads_ids_sorted, ads_id)
                if idx < len(existed_ads_ids_sorted) and existed_ads_ids_sorted[idx] == ads_id:
                    continue
            all_links.append((link, ngay_gia_han))

    all_links = list(set(all_links))
    print(f"scrape_links_from_base -> found {len(all_links)} links")
    return all_links


def determine_start_date():
    max_ngay_gia_han = get_max_ngay_gia_han()

    if max_ngay_gia_han:
        # Add 1 day to the max date
        start_date = max_ngay_gia_han + timedelta(days=1)
    else:
        # Raise an error if the table is empty
        raise Exception("The table is empty")

    return start_date

def scrape_links_wrapper():
    base_urls = [
        "https://batdongsan.com.vn/nha-dat-ban/p{i}",
        "https://batdongsan.com.vn/nha-dat-cho-thue/p{i}"
    ]

    # start_date = determine_start_date()
    start_date = date.today() - timedelta(days=1)
    end_date = date.today() - timedelta(days=1)

    all_scraped_links = []
    # existed_ads_ids = get_existed_ads_ids()
    existed_ads_ids = None

    for base in base_urls:
        # max_page = get_max_page_with_retries(base, attempts=5, verbose=True)
        soup, _ = soup_from_url(base.format(i=1))
        
        BDS_count = soup.find("span", class_="re__srp-total-count")
        if BDS_count:
            print(BDS_count.text)
            max_page = round(int(re.sub(r"\.", "", re.search(r"[\d\.]+", BDS_count.text).group()))/20)
        if max_page is None or max_page <= 0:
            raise
        else:
            print("URL: ", base)
            print("max_page = ", max_page)
        
        # phase 1
        links, min_page = scrape_links_from_base_phase1(base, start_date, end_date, max_page,
                                       existed_ads_ids_sorted=None if existed_ads_ids is None else [int(x) for x in existed_ads_ids])
        all_scraped_links.extend(links)

        # phase 2
        start_page = find_start_page(base, end_date, min_page, max_page)
        print(f"For base {base}, determined start_page={start_page}")
        links = scrape_links_from_base_phase2(base, start_page, start_date, end_date, max_page,
                                       existed_ads_ids_sorted=None if existed_ads_ids is None else [int(x) for x in existed_ads_ids])
        if len(links) < 1000:
            raise Exception("Links scraping failed.")
        all_scraped_links.extend(links)
        print()

    # save to CSV in Lakehouse
    all_scraped_links = list(set(all_scraped_links))
    os.makedirs(ADS_DATA_DIR, exist_ok=True)
    scraped_csv = os.path.join(ADS_DATA_DIR, "scraped_links.csv")
    if os.path.exists(scraped_csv):
        os.remove(scraped_csv)
    with open(scraped_csv, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        for link, ngay in all_scraped_links:
            writer.writerow([link, ngay.isoformat()])
    print(f"Number of URLs: {len(all_scraped_links)}")

In [13]:
# --- Parse detail page (single-threaded) ---

def parse_ad_page(url, resp_text=None):
    try:
        if resp_text is None:
            raise
        else:
            soup = BeautifulSoup(resp_text, "html.parser")
        ad_data = {}

        # Tìm mã tin
        ads_id = extract_ads_id_from_link(url)
        if ads_id:
            ad_data["Mã tin"] = str(ads_id)

        # Tìm ...
        try:
            config_items = soup.select(".js__pr-config-item")
            for item in config_items:
                title_el = item.select_one(".title")
                value_el = item.select_one(".value")
                if title_el and value_el:
                    ad_data[title_el.get_text(strip=True)] = value_el.get_text(strip=True)
        except Exception:
            print(f"Error parsing config items for {url}")

        # Tìm ...
        try:
            bc = soup.select_one(".js__ob-breadcrumb")
            if bc:
                a_tags = bc.find_all("a")
                if len(a_tags) >= 1:
                    ad_data["Loại quảng cáo"] = a_tags[0].get_text(strip=True)
                    ad_data["Loại BĐS"] = a_tags[0].get("title") or a_tags[0].get_text(strip=True)
                if len(a_tags) >= 2:
                    ad_data["Tỉnh, thành phố"] = a_tags[1].get_text(strip=True)
                if len(a_tags) >= 3:
                    ad_data["Quận"] = a_tags[2].get_text(strip=True)
                if len(a_tags) >= 4:
                    ad_data["Địa chỉ 1"] = a_tags[3].get_text(strip=True)
        except Exception:
            print(f"Error parsing breadcrumbs for {url}")

        # Tìm Địa chỉ 2
        try:
            addr_el = soup.find(class_="js__pr-address")
            if addr_el:
                ad_data["Địa chỉ 2"] = addr_el.get_text(" ", strip=True)
        except Exception:
            print(f"Error parsing address for {url}")

        # Tìm đặc điểm bất động sản
        try:
            titles = soup.select(".re__pr-specs-content-item-title")
            values = soup.select(".re__pr-specs-content-item-value")
            for t, v in zip(titles, values):
                ad_data[t.get_text(strip=True)] = v.get_text(strip=True)
        except Exception:
            print(f"Error parsing product features for {url}")

        # Tìm dự án
        try:
            container = soup.find(class_="re__project-card-info") or soup.find(class_="re__ldp-project-info")
            if not container:
                ad_data["Link dự án"] = None
                ad_data["Tên dự án"] = None
            else:
                title_tag = container.find("div", class_="re__project-title")
                ad_data["Tên dự án"] = title_tag.get_text(strip=True) if title_tag else None
                ad_data["Link dự án"] = None
                avatar = container.find("div", class_="re__section-avatar")
                if avatar:
                    a = avatar.find("a", href=True)
                    if a:
                        ad_data["Link dự án"] = a["href"]
        except Exception:
            print(f"Error parsing project info for {url}")

        # Tìm tọa độ
        try:
            coords = None
            map_block = soup.select_one(".re__pr-map, .re__map")
            if map_block:
                img = map_block.find("img", attrs={"data-src": True})
                if img:
                    ds = img["data-src"]
                    m = re.search(r"center=([\d\.-]+),([\d\.-]+)", ds)
                    if m:
                        coords = f"{m.group(1)},{m.group(2)}"
                    else:
                        m2 = re.search(r"=([\d\.-]+),([\d\.-]+)&", ds)
                        if m2:
                            coords = f"{m2.group(1)},{m2.group(2)}"
                        else:
                            m3 = re.search(r"([\d\.-]+),([\d\.-]+)", ds)
                            if m3:
                                coords = f"{m3.group(1)},{m3.group(2)}"
                if not coords:
                    iframe = map_block.find("iframe", class_="lazyload")
                    if iframe:
                        url = iframe.get("data-src") or iframe.get("src")
                        if not url:
                            coords = ""
                        url = unquote(url)
                        parsed = urlparse(url)
                        qs = parse_qs(parsed.query)
                        q_vals = qs.get("q")
                        if q_vals:
                            q0 = q_vals[0]
                            m = re.search(r"(-?\d+\.\d+)\s*,\s*(-?\d+\.\d+)", q0)
                            if m:
                                coords = f"{m.group(1)},{m.group(2)}"
                        m = re.search(r"@(-?\d+\.\d+),\s*(-?\d+\.\d+)", url)
                        if m:
                            coords = f"{m.group(1)},{m.group(2)}"
                        m = re.search(r"(-?\d+\.\d+)\s*,\s*(-?\d+\.\d+)", url)
                        if m:
                            coords = f"{m.group(1)},{m.group(2)}"
            ad_data["Tọa độ"] = coords or ""
        except Exception:
            print(f"Error parsing map coords for {url}")
            ad_data["Tọa độ"] = ""

        # Tìm ngày đăng
        try:
            labels = soup.select(".re__pr-publish-info, .pr-publish")
            if labels:
                txt = " ".join([l.get_text(" ", strip=True) for l in labels])
                m = re.search(r"(\d{1,2}/\d{1,2}/\d{4})", txt)
                if m:
                    ad_data["Ngày đăng"] = m.group(1)
        except Exception:
            print(f"Error parsing publish dates for {url}")

        # Tìm số tầng
        try:
            ad_title_el = soup.select_one(".re__pr-title")
            ad_desc_el = soup.select_one(".re__detail-content")

            ad_title = ad_title_el.get_text(" ", strip=True).lower() if ad_title_el else ""
            ad_desc = ad_desc_el.get_text(" ", strip=True).lower() if ad_desc_el else ""

            ad_data["Tiêu đề"] = ad_title
            ad_data["Mô tả"] = ad_desc

            # Giá trị mặc định
            ad_data.setdefault("Số tầng lookup", None)

            prop_type = ad_data.get("Loại BĐS", "")

            allowed_types = {"Bán nhà riêng tại Việt Nam", "Bán nhà biệt thự, liền kề tại Việt Nam", "Bán nhà mặt phố tại Việt Nam"}

            def _find_floors_in_text(text: str):
                if not text:
                    return None

                # Nhà cấp 4 / nhà c4 -> 1 tầng
                if re.search(r'\bnhà\s*(cấp\s*)?4\b', text) or re.search(r'\bnhà\s*c4\b', text):
                    return 1

                # Tìm "x lầu" hoặc "xlầu" => x + 1
                m = re.search(r'(\d+)\s*lầu\b', text)
                if not m:
                    m = re.search(r'(\d+)lầu\b', text)
                if m:
                    try:
                        return int(m.group(1)) + 1
                    except ValueError:
                        pass

                # Tìm "x tầng" hoặc "xtầng" => x
                m = re.search(r'(\d+)\s*tầng\b', text)
                if not m:
                    m = re.search(r'(\d+)tầng\b', text)
                if m:
                    try:
                        return int(m.group(1))
                    except ValueError:
                        pass

                # Một vài trường hợp khác: "tầng x" (ví dụ "tầng 3")
                m = re.search(r'tầng\s*(\d+)\b', text)
                if m:
                    try:
                        return int(m.group(1))
                    except ValueError:
                        pass

                return None

            # Chỉ tiến hành tìm nếu thuộc 1 trong 3 loại và key "Số tầng" chưa có giá trị
            existing_floor_value = ad_data.get("Số tầng")
            if prop_type in allowed_types and not existing_floor_value:
                floors = _find_floors_in_text(ad_title)
                if floors is None:
                    floors = _find_floors_in_text(ad_desc)

                # Gán kết quả
                ad_data["Số tầng lookup"] = floors

                # Nếu chưa có "Số tầng" gốc thì cập nhật luôn
                if floors is not None and not existing_floor_value:
                    ad_data["Số tầng"] = str(floors) + " tầng"

        except Exception:
            print(f"Error finding Số tầng for {url}")

        # Tìm căn góc
        can_goc = 1 if (
            (ad_title and "góc" in ad_title.lower()) or
            (ad_desc and "góc" in ad_desc.lower())
        ) else 0
        ad_data["Căn góc"] = can_goc
        
        return ad_data
    except Exception as e:
        print(f"[ERROR parse_ad_page] {url} -> {e}")
        return None

In [14]:
def collect_ads_data_wrapper():
    os.makedirs(ADS_DATA_DIR, exist_ok=True)
    worker_dir = os.path.join(ADS_DATA_DIR, "worker0")
    os.makedirs(worker_dir, exist_ok=True)

    # load links from scraped_links.csv
    links = []
    scraped_csv = os.path.join(ADS_DATA_DIR, "scraped_links.csv")
    if os.path.exists(scraped_csv):
        with open(scraped_csv, mode="r", encoding="utf-8") as f:
            reader = csv.reader(f)
            for row in reader:
                if not row:
                    continue
                url = row[0].strip()
                try:
                    dt = datetime.strptime(row[1].strip(), "%Y-%m-%d").date()
                except Exception:
                    dt = None
                links.append((url, dt))

    # detect already-saved batches and count processed rows so we can resume
    parquet_pattern = os.path.join(worker_dir, "ads_data_batch*.parquet")
    existing_files = sorted(glob.glob(parquet_pattern))
    processed_rows = 0
    saved_batches = 0
    for p in existing_files:
        try:
            df_existing = pd.read_parquet(p)
            processed_rows += len(df_existing)
            saved_batches += 1
        except Exception as e:
            print(f"[WARN] can't read existing parquet {p}: {e}")

    if processed_rows > 0:
        print(f"[INFO] Resuming: found {saved_batches} saved batch(es) with {processed_rows} total rows. "
              f"Skipping first {processed_rows} links from scraped_links.csv.")

    batch = []
    # start from processed_rows (assumes previous run processed links in the same order)
    start_idx = processed_rows
    total_links = len(links)

    for global_idx in range(start_idx, total_links):
        url, ngay_gia_han = links[global_idx]
        try:
            resp = get_with_retry(url)
            ad = parse_ad_page(url, resp_text=resp.text)
            if not ad:
                print(f"parse returned no data for {url}")
                continue
            ad["Ngày gia hạn"] = ngay_gia_han
            if "Mã tin" not in ad or not ad["Mã tin"]:
                mid = extract_ads_id_from_link(url)
                if mid:
                    ad["Mã tin"] = str(mid)
            batch.append(ad)

            if len(batch) >= BATCH_SAVE_SIZE:
                df = pd.DataFrame(batch)
                saved_batches += 1
                filename = os.path.join(worker_dir, f"ads_data_batch{saved_batches}.parquet")
                df.to_parquet(filename, index=False)
                # print(f"[INFO] Saved batch #{saved_batches}: {filename} (rows: {len(df)})")
                batch = []
        except Exception as e:
            print(f"[WARN worker0] {url} -> {e}")
            continue

    # save leftover
    if batch:
        df = pd.DataFrame(batch)
        saved_batches += 1
        filename = os.path.join(worker_dir, f"ads_data_batch{saved_batches}.parquet")
        df.to_parquet(filename, index=False)
        # print(f"[INFO] Saved final batch #{saved_batches}: {filename} (rows: {len(df)})")

In [15]:
# --- Helpers: clear previous data (lakehouse worker dir) ---

def clear_previous_data():
    worker_dir = os.path.join(ADS_DATA_DIR, "worker0")
    if os.path.exists(worker_dir):
        for fname in os.listdir(worker_dir):
            fpath = os.path.join(worker_dir, fname)
            try:
                if os.path.isfile(fpath):
                    os.remove(fpath)
                elif os.path.isdir(fpath):
                    shutil.rmtree(fpath)
            except Exception as e:
                print(f"Failed to remove {fpath}: {e}")
                
    os.makedirs(worker_dir, exist_ok=True)

In [16]:
def print_random_values(df, n=10):
    for col in df.columns:
        print(f"\n--- {col} ---")
        print(df[col].sample(min(n, len(df)), random_state=42).to_list())
    print()

In [17]:
# --- Data processing & push to BigQuery (đọc từ Lakehouse parquet files) ---

def data_processing(df1: pd.DataFrame):
    df = df1.copy()

    def convert_to_float(text, unit=" m²"):
        try:
            if pd.isna(text):
                return None
            text = str(text)
            text = text.replace(".", "").replace(",", ".")
            text = text.replace(unit, "").strip()
            return float(text)
        except:
            return None

    if "Diện tích" in df.columns:
        df["Diện tích"] = df["Diện tích"].apply(lambda x: convert_to_float(x, unit="m²") if isinstance(x, str) else convert_to_float(x))
    
    if "Số phòng ngủ" in df.columns:
        df["Số phòng ngủ"] = df["Số phòng ngủ"].astype(str).str.replace(" phòng", "", regex=False).replace({"nan": None})
        df["Số phòng ngủ"] = pd.to_numeric(df["Số phòng ngủ"], errors="coerce").astype("Int64")
    
    if "Số phòng tắm, vệ sinh" in df.columns:
        df["Số phòng tắm, vệ sinh"] = df["Số phòng tắm, vệ sinh"].astype(str).str.replace(" phòng", "", regex=False).replace({"nan": None})
        df["Số phòng tắm, vệ sinh"] = pd.to_numeric(df["Số phòng tắm, vệ sinh"], errors="coerce").astype("Int64")
    
    if "Số toilet" in df.columns:
        df["Số toilet"] = df["Số toilet"].astype(str).str.replace(" phòng", "", regex=False).replace({"nan": None})
        df["Số toilet"] = pd.to_numeric(df["Số toilet"], errors="coerce").astype("Int64")
    
    if "Số tầng lookup" in df.columns:
        df["Số tầng lookup"] = df["Số tầng lookup"].astype("Int64")

    # Process dates
    for col in ["Ngày đăng", "Ngày gia hạn", "Ngày hết hạn"]:
        if col in df.columns:
            try:
                df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")
            except:
                df[col] = pd.to_datetime(df[col], errors="coerce")

    # Process "Loại BĐS"
    if "Loại BĐS" in df.columns:
        df["Loại BĐS"] = df["Loại BĐS"].astype(str).str.replace("Bán ", "", regex=False)
        df["Loại BĐS"] = df["Loại BĐS"].astype(str).str.replace("Cho thuê ", "", regex=False)
        df["Loại BĐS"] = df["Loại BĐS"].astype(str).str.replace("Cho thuê, sang nhượng ", "", regex=False)
        df["Loại BĐS"] = df["Loại BĐS"].astype(str).str.replace(" tại Việt Nam", "", regex=False)

    # Process "Địa chỉ 1"
    if "Địa chỉ 1" in df.columns:
        def process_khu_vuc(text):
            try:
                return text.split(" tại ")[1]
            except:
                return None
        df["Địa chỉ 1"] = df["Địa chỉ 1"].apply(lambda x: process_khu_vuc(x) if isinstance(x, str) else None)

    # Process "Địa chỉ 2"
    if "Địa chỉ 2" in df.columns:
        pattern = re.compile(r'(Phường|P\.?\s*\d+|P\b|Xã|Xa|Thị trấn|TT)', re.I)
        def extract_phuong(text):
            if not isinstance(text, str):
                return None
            parts = [p.strip() for p in text.split(",")]
            for p in parts:
                if pattern.search(p):
                    return p
            return None
        df["Phường/Xã/Thị trấn"] = df["Địa chỉ 2"].apply(lambda x: extract_phuong(x) if isinstance(x, str) else None)

    # Process price
    if "Khoảng giá" in df.columns:
        # helper: lấy các số trong chuỗi, trả trung bình nếu có nhiều số (ví dụ "10-12")
        def _extract_number_mean(s):
            if pd.isna(s):
                return None
            s = str(s)
            nums = re.findall(r'\d+(?:[\.,]\d+)?', s)
            if not nums:
                return None
            vals = []
            for n in nums:
                # bỏ dấu ngăn phần nghìn nếu có, đổi dấu phẩy thành chấm
                n = n.replace('.', '').replace(',', '.')
                try:
                    vals.append(float(n))
                except:
                    continue
            if not vals:
                return None
            return sum(vals) / len(vals)

        def _parse_price_row(row):
            raw = row.get("Khoảng giá", "")
            if pd.isna(raw) or str(raw).strip() == "":
                return np.nan
            text = str(raw).lower()

            # số (hoặc trung bình nếu range)
            number = _extract_number_mean(text)
            if number is None:
                # có thể là chữ 'thỏa thuận'...
                if "thỏa" in text or "thoả" in text:
                    return np.nan
                return np.nan

            # xác định hệ số (tỷ/triệu/nghìn)
            if "tỷ" in text:
                factor = 1e9
            elif "triệu" in text:
                factor = 1e6
            elif "nghìn" in text or "nghin" in text:
                factor = 1e3
            else:
                # nếu không thấy unit, giữ nguyên số (giả sử đã là VND)
                factor = 1

            # kiểm tra xem giá có phải là giá trên m2 hay không
            if ("/m" in text) or ("m2" in text) or ("m²" in text):
                price_per_m2 = number * factor
                # cần có cột Diện tích để nhân
                if "Diện tích" in row.index:
                    area = row.get("Diện tích")
                    # area = _extract_number_mean(area_raw)
                    if area is None:
                        return np.nan
                    return price_per_m2 * area
                else:
                    # không có cột diện tích -> không thể nhân
                    return np.nan
            else:
                # giá tuyệt đối
                return number * factor
        df["Khoảng giá"] = df.apply(_parse_price_row, axis=1)

    # Process Tọa độ
    if "Tọa độ" in df.columns:
        def split_coords(x):
            try:
                if not x or x == "":
                    return (None, None)
                if isinstance(x, str):
                    parts = x.split(",")
                    if len(parts) >= 2:
                        return (float(parts[0]), float(parts[1]))
                return (None, None)
            except:
                return (None, None)
        coords = df["Tọa độ"].apply(lambda x: split_coords(x))
        df["Tọa độ x"] = coords.apply(lambda t: t[0])
        df["Tọa độ y"] = coords.apply(lambda t: t[1])
        df = df.drop(columns=["Tọa độ"])

    # column_mapping
    column_mapping = {
        "Phường/Xã/Thị trấn": "Phường Xã Thị trấn", "Tỉnh, thành phố": "Tỉnh thành phố", "Số phòng tắm, vệ sinh": "Số phòng tắm - vệ sinh"
    }
    df.rename(columns=column_mapping, inplace=True)
    
    # required columns
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str)
    needed = ["Mã tin", "Loại tin", "Ngày đăng", "Ngày hết hạn", "Loại quảng cáo", "Loại BĐS", "Tỉnh thành phố",
              "Quận", "Diện tích", "Địa chỉ 1"]
    existing_needed = [c for c in needed if c in df.columns]
    if existing_needed:
        df.dropna(subset=existing_needed, inplace=True)
    df = df.replace(['nan', "None", 'NA', 'N/A', 'null'], np.nan).infer_objects(copy=False)

    # remove control characters 
    df[["Tiêu đề", "Mô tả"]] = df[["Tiêu đề", "Mô tả"]].apply(
        lambda col: col.str.replace(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', regex=True))
    return df


from google.oauth2 import service_account

def push_data_to_bigquery(
    lakehouse_dir=ADS_DATA_DIR,
    project_id="real-estate-project-445516",
    dataset_id="real_estate_data",
    table_id="ads_data",
    json_key_path="/lakehouse/default/Files/real-estate-project-445516-83dc50b692bc.json",   # path to service account json
    overwrite=True        # if True -> WRITE_TRUNCATE, else -> WRITE_APPEND
):
    # read worker0 parquet files (same as your original)
    df_all = pd.DataFrame()
    worker_dir = os.path.join(lakehouse_dir, "worker0")
    if not os.path.exists(worker_dir):
        print(f"No worker data found in {worker_dir}")
        return

    files = [os.path.join(worker_dir, f) for f in os.listdir(worker_dir) if f.endswith(".parquet")]
    for path in files:
        try:
            df_temp = pd.read_parquet(path)
            df_all = pd.concat([df_all, df_temp], ignore_index=True)
        except Exception as e:
            print(f"[WARN] Failed to read {path}: {e}")
            continue

    if df_all.empty:
        print("No data to push to BigQuery.")
        return

    # check data values
    print_random_values(df_all)

    # apply your data processing function
    df_proc = data_processing(df_all)
    df_proc.to_excel(os.path.join(lakehouse_dir, "worker0/data.xlsx"), index=False)
    print("Dataframe info:\n%s", df_proc.info())

    # create BigQuery client with optional service account credentials
    if json_key_path:
        if not os.path.exists(json_key_path):
            raise FileNotFoundError(f"JSON key not found: {json_key_path}")
        creds = service_account.Credentials.from_service_account_file(json_key_path)
        client = bigquery.Client(project=project_id, credentials=creds)
    else:
        # will use Application Default Credentials if available
        client = bigquery.Client(project=project_id)

    table_ref = f"{project_id}.{dataset_id}.{table_id}"
    write_disp = "WRITE_TRUNCATE" if overwrite else "WRITE_APPEND"
    job_config = bigquery.LoadJobConfig(write_disposition=write_disp)
    if not overwrite:
        job_config.schema_update_options = ["ALLOW_FIELD_ADDITION"]
    
    # Check for schema mismatch
    table = client.get_table(table_ref)  # raises if not exists
    dest_cols = {f.name: f.field_type for f in table.schema}
    df_cols = list(df_proc.columns)
    extra = [c for c in df_cols if c not in dest_cols]
    missing = [c for c in dest_cols if c not in df_cols]
    if extra:
        raise ValueError(f"DataFrame has extra columns: {extra}")
    exceptions = [
        'Địa chỉ 2',
        'Thời gian dự kiến vào ở',
        'Mức giá nước',
        'Mức giá internet',
        'Tiện ích',
        'Mức giá điện',
        'Phường Xã Thị trấn'
    ]
    missing = [col for col in missing if col not in exceptions]
    if missing:
        raise ValueError(f"DataFrame has missing columns: {missing}")

    # Write
    try:
        job = client.load_table_from_dataframe(df_proc, table_ref, job_config=job_config)
        job.result()  # wait for completion
        print(f"Successfully pushed data to {table_ref} (overwrite={overwrite})")
    except Exception as e:
        print(f"Error pushing to BigQuery: {e}")
        raise

In [18]:
import pandas as pd

def print_unique_values(df, columns, max_values=100):
    if isinstance(columns, str):
        columns = [columns]  # Cho phép truyền 1 tên cột duy nhất

    for col in columns:
        if col not in df.columns:
            print(f"⚠️ Column '{col}' not found in DataFrame.\n")
            continue

        unique_vals = df[col].dropna().unique()
        total = len(unique_vals)
        print(f"🔹 Unique values in column '{col}' ({total} total):")

        # Giới hạn in ra tối đa max_values
        if total > max_values:
            print(unique_vals[:max_values])
            print(f"... (showing first {max_values} of {total} unique values)")
        else:
            print(unique_vals)

        print("-" * 80)

# df = pd.read_excel("/lakehouse/default/Files/ads_data/worker0/data.xlsx")
# print_unique_values(df, ['Thời gian dự kiến vào ở', 'Mức giá điện', 'Mức giá nước', 'Mức giá internet', 'Tiện ích'])

In [19]:
session = create_session()
clear_previous_data()
scrape_links_wrapper()
collect_ads_data_wrapper()
push_data_to_bigquery(overwrite=False)


--- Mã tin ---
['45228074', '45002110', '44351994', '44704144', '44243728', '45215506', '45225857', '43900711', '45206303', '44484415']

--- Ngày đăng ---
['04/03/2026', '04/03/2026', '18/02/2026', '25/02/2026', '27/02/2026', '02/03/2026', '04/03/2026', '04/03/2026', '01/03/2026', '04/03/2026']

--- Ngày hết hạn ---
['19/03/2026', '19/03/2026', '06/03/2026', '12/03/2026', '14/03/2026', '02/04/2026', '11/03/2026', '19/03/2026', '08/03/2026', '19/03/2026']

--- Loại tin ---
['Tin thường', 'Tin thường', 'Tin thường', 'Tin thường', 'Tin thường', 'Tin thường', 'Tin VIP Bạc', 'Tin thường', 'Tin VIP Bạc', 'Tin thường']

--- Loại quảng cáo ---
['Bán', 'Bán', 'Bán', 'Bán', 'Bán', 'Bán', 'Cho thuê', 'Bán', 'Bán', 'Bán']

--- Loại BĐS ---
['Bán căn hộ chung cư tại Việt Nam', 'Bán nhà riêng tại Việt Nam', 'Bán căn hộ chung cư tại Việt Nam', 'Bán nhà riêng tại Việt Nam', 'Bán căn hộ chung cư tại Việt Nam', 'Bán nhà biệt thự, liền kề tại Việt Nam', 'Cho thuê căn hộ chung cư tại Việt Nam', 'Bán nh